# 03 – Inference Demo

Load a trained model and run inference on sample images.
Shows overlay, probability map, uncertainty map and metrics.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from inference.engine import SegmentationEngine
from training.visualization import plot_segmentation

%matplotlib inline

In [ ]:
engine = SegmentationEngine.from_config('../configs/config.yaml')
print(f'Engine ready on: {engine.device}')

In [ ]:
# Replace with a real image path after training
sample_images = sorted(Path('../data/raw/images').glob('*.png'))[:4]
sample_masks  = sorted(Path('../data/raw/masks').glob('*.png'))[:4]

if not sample_images:
    print('No images found. Run:  python scripts/generate_samples.py')
else:
    for img_p, msk_p in zip(sample_images, sample_masks):
        result = engine.predict_file(img_p, gt_mask_path=msk_p)

        metrics = result.get('metrics', {})
        title = (f"{img_p.stem} | "
                 f"tumor={result['tumor_percentage']:.1f}% | "
                 f"conf={result['confidence']:.2f}" +
                 (f" | dice={metrics['dice']:.3f}" if metrics else ''))

        plot_segmentation(
            image     = result['processed_image'],
            gt_mask   = engine.preprocessor.process_mask(engine.preprocessor.load_mask(msk_p)),
            pred_mask = result['pred_mask'],
            pred_prob = result['prob_map'],
            title     = title,
        )

In [ ]:
# Uncertainty map demo (MC-Dropout)
if sample_images:
    result = engine.predict_file(sample_images[0])
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax, (data, title, cmap) in zip(axes, [
        (result['processed_image'].squeeze(), 'MRI',         'gray'),
        (result['pred_mask'],                 'Prediction',  'gray'),
        (result['uncertainty_map'],           'Uncertainty', 'plasma'),
    ]):
        ax.imshow(data, cmap=cmap)
        ax.set_title(title); ax.axis('off')

    plt.tight_layout()
    plt.show()